# Drift detection robustness sweep

How large does a change have to be before a drift detector notices it, and what
does that cost in false alarms?

This notebook sweeps **four drift shapes x five magnitudes x four detectors** on
SKAB's anomaly-free recording. That is 80 streaming runs — tedious on a laptop,
a few minutes on free Kaggle CPU. No GPU is needed; `river` is pure streaming
code.

The four detectors are ADWIN and KSWIN (both watch the signal's level),
`adwin_var` (watches its variance), and `adwin_meanvar` (fires when either the
mean or the variance detector does). The variance-based pair exist because the
mean detectors are flat against magnitude on gradual drift; see the closing notes.

## The control that makes this worth running

Every sweep includes **magnitude 0.0**: no drift injected at all. This is the
whole point of the notebook rather than a formality.

SKAB's anomaly-free recording has no annotated faults, but it is **not
stationary** — the process genuinely drifts on its own. A detector fires on it
regardless. Without the control those firings get silently credited to the
injected change, and the method looks far better than it is.

Read `excess_firings` (firings above the 0.0 control) and **detection delay**,
not raw detection counts.

## Requirements

Either of these is enough:

- **Internet enabled** (Settings -> Internet). The loader fetches SKAB from its
  public GitHub repository.
- **A SKAB dataset attached** to the notebook. Any dataset containing SKAB's
  `anomaly-free.csv` works; the next cell searches `/kaggle/input` for it.

Outputs are written to `/kaggle/working`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORKING = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
REPO_URL = "https://github.com/DhruvilShah22/drift-aware-anomaly-detection.git"

print("running on Kaggle" if ON_KAGGLE else "running locally")

# The sweep lives in the repo (src/sweep.py) rather than being pasted in here,
# so the notebook runs the same tested code as everything else. Locally the repo
# is already the working directory; on Kaggle it gets cloned.
if ON_KAGGLE:
    repo = Path("/kaggle/working/drift-aware-anomaly-detection")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    project_root = repo
else:
    project_root = Path.cwd()
    if not (project_root / "src").exists():
        project_root = project_root.parent  # started from notebooks/

sys.path.insert(0, str(project_root))
print("project root:", project_root)

In [ ]:
!pip install -q "river>=0.25,<0.26"

## Locate SKAB

Prefer an attached Kaggle dataset; fall back to downloading from GitHub. Either
way the frame is verified — expected sensor columns present, no missing values —
before anything is built on it.

In [ ]:
import pandas as pd

from src.data_loader import SENSOR_COLUMNS, StreamData, load_anomaly_free


def find_attached_skab() -> Path | None:
    """Look for SKAB's anomaly-free recording in any attached Kaggle dataset."""
    root = Path("/kaggle/input")
    if not root.exists():
        return None
    for candidate in root.rglob("*.csv"):
        if "anomaly-free" in candidate.name.lower():
            return candidate
    return None


attached = find_attached_skab()

if attached is not None:
    print(f"using attached dataset: {attached}")
    frame = pd.read_csv(attached, sep=";", index_col="datetime", parse_dates=True)
    missing = [c for c in SENSOR_COLUMNS if c not in frame.columns]
    if missing:
        raise SystemExit(
            f"{attached} is missing expected SKAB columns {missing}. "
            "Stopping rather than guessing at a different schema."
        )
    sensors = frame[SENSOR_COLUMNS].astype(float)
    base = StreamData(
        name=attached.name,
        frame=sensors,
        labels=pd.Series(0, index=frame.index, dtype=int),
    )
else:
    print("no attached SKAB dataset found; downloading from GitHub")
    base = load_anomaly_free()

print(base.describe())
assert not base.frame.isna().any().any(), "SKAB frame contains missing values"

## Run the sweep

80 runs. Roughly 6-8 minutes on Kaggle CPU.

In [ ]:
import time

from src.sweep import SweepConfig, attribution_table, summarise_sweep, sweep

config = SweepConfig(
    magnitudes=(0.0, 1.0, 2.0, 3.0, 5.0),
    detectors=("adwin", "kswin", "adwin_var"),
    warmup_size=1000,
)

started = time.perf_counter()
table = sweep(base, config)
table = attribution_table(table)
print(f"\nfinished {len(table)} runs in {time.perf_counter() - started:.0f}s")

table.to_csv(WORKING / "sweep.csv", index=False)
print("wrote", WORKING / "sweep.csv")

## Results

### Detection delay against magnitude

Delay is the metric that carries information here. Detection *rate* does not:
see the control check below.

In [ ]:
summary = summarise_sweep(table)
summary.to_csv(WORKING / "sweep_summary.csv", index=False)
summary

In [ ]:
import numpy as np

delay = (
    table[table.magnitude > 0]
    .pivot_table(index="magnitude", columns=["detector", "kind"], values="mean_delay")
)
delay.round(0)

### The control check

`detection_rate` is 1.0 at every magnitude **including 0.0**, where nothing was
injected. The base recording drifts enough on its own that some firing always
lands within the matching horizon of an arbitrary point. Reported alone, that
number would be meaningless.

`excess_firings` — firings above the 0.0 control — is the honest measure of what
the injected change actually caused.

In [ ]:
control_check = table[["kind", "magnitude", "detector", "detection_rate",
                       "mean_delay", "n_detections_total", "control_firings",
                       "excess_firings"]]
control_check.sort_values(["kind", "detector", "magnitude"])

In [ ]:
import matplotlib.pyplot as plt

kinds = sorted(table.kind.unique())
fig, axes = plt.subplots(1, len(kinds), figsize=(4 * len(kinds), 3.6), sharey=True)

for ax, kind in zip(np.atleast_1d(axes), kinds):
    subset = table[table.kind == kind]
    for detector, colour in (("adwin", "#2a7f4f"), ("kswin", "#1f6f8b"),
                             ("adwin_var", "#c1440e"), ("adwin_meanvar", "#6a3d9a")):
        rows = subset[subset.detector == detector].sort_values("magnitude")
        ax.plot(rows.magnitude, rows.mean_delay, marker="o", color=colour, label=detector)
    ax.set_title(kind, fontsize=10)
    ax.set_xlabel("injected magnitude (sd)")
    ax.grid(alpha=0.25)

np.atleast_1d(axes)[0].set_ylabel("mean detection delay (rows)")
np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
fig.suptitle(
    "Detection delay against change magnitude\n"
    "magnitude 0.0 is the control: no drift injected, so its delay is the "
    "background rate of the recording's own drift",
    fontsize=9,
)
fig.tight_layout()
fig.savefig(WORKING / "sweep_delay.png", dpi=140)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for detector, colour in (("adwin", "#2a7f4f"), ("kswin", "#1f6f8b"),
                         ("adwin_var", "#c1440e"), ("adwin_meanvar", "#6a3d9a")):
    rows = summary[summary.detector == detector]
    ax.plot(rows.magnitude, rows.false_alarms_per_1k, marker="s", color=colour,
            label=f"{detector} false alarms / 1k")
ax.set_xlabel("injected magnitude (sd)")
ax.set_ylabel("false alarms per 1000 steps")
ax.set_title(
    "False-alarm rate by detector\n"
    "(upper bound: the base recording's own drift counts against this)",
    fontsize=10, loc="left",
)
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(WORKING / "sweep_false_alarms.png", dpi=140)
plt.show()

## What this sweep shows

Filled in from a real run of exactly this code (see `results/sweep.csv` in the
repo):

1. **ADWIN reacts far faster once a change exists.** On the sudden step its mean
   delay falls from 393 rows with nothing injected to **9 rows** at 1 sd and
   stays there — the change is caught essentially immediately, and making it
   larger does not help because it was already saturated.
2. **Incremental drift is where magnitude genuinely matters.** ADWIN's delay
   falls monotonically, 373 -> 277 -> 245 -> 181 -> 117 rows as the ramp
   steepens. A slow ramp is intrinsically harder to catch than a step.
3. **Gradual drift defeats magnitude.** Delay sits near 245 rows regardless of
   size, because early in the transition the new regime looks like sporadic
   outliers rather than a shift in level.
4. **ADWIN beats KSWIN on both axes** — shorter delays and roughly 1.9 vs 2.8
   false alarms per 1000 steps.
5. **`adwin_var` cracks gradual drift.** Watching variance instead of level, its
   gradual delay falls 213 -> 117 -> 85 -> 85 as magnitude rises (ADWIN is flat
   at 245) and it is the quietest of all four at 0.28 false alarms per 1000
   steps — but it is **blind to incremental drift** (a smooth ramp changes the
   mean, not the spread), detecting 19 of 24 change points.
6. **`adwin_meanvar` catches every shape at almost no extra cost.** Firing when
   either the mean or the variance ADWIN does, it detects 24 of 24 change points,
   matches ADWIN on sudden/incremental/recurring and adwin_var's best gradual
   delay (beating it at low magnitude, 149 vs 213 at 1 sd), at 1.98 false alarms
   per 1000 steps against ADWIN's 1.93 — the variance branch's false alarms land
   on the same background drift ADWIN already fires on, so the union is not the
   sum. It is the best all-round detector here.
7. **Detection rate is uninformative on this data.** It is 1.0 everywhere,
   including the control. Any claim resting on it would be worthless.

The honest limitation: because the base recording drifts on its own,
`excess_firings` is near zero or negative in most conditions. Injecting drift
changes *when* the detector fires much more than *how often*. That is a real
constraint of evaluating on this dataset, not a result to paper over.

## Event-level anomaly quality (labelled SKAB valve1)

The sweep above measures drift-detection *timing* on the anomaly-free recording,
which by construction has no labelled anomalies, so anomaly F1 cannot be computed
there. Anomaly quality needs the labelled recordings and it needs the right
metric.

On SKAB valve1 the fault blocks are long and contiguous (15 events, ~390 rows
each). Point-wise F1 rewards flagging every row of a block, so it buries a
sparse-but-correct detector and clusters every strategy near the flag-everything
baseline (F1 ~0.51 at this 34% base rate). Scoring each fault block as one
**event** -- recall becomes "was each fault noticed at all", precision stays
point-wise as the volume penalty -- pulls them apart. The metric is
`event_metrics` in `src/evaluate.py`.

This runs the same tested `compare()` used everywhere else. It needs SKAB's
labelled valve1 recordings, so **internet must be enabled** (or a SKAB dataset
containing the valve1 group attached).

In [ ]:
from src.experiment import compare, default_strategies, labelled_scenario

labelled = labelled_scenario()
print(f"{labelled.name}: {len(labelled.frame)} rows, "
      f"{labelled.labels.mean():.1%} anomalous")

labelled_table, _ = compare(labelled, default_strategies(), verbose=True)

event_view = labelled_table[[
    "strategy", "f1", "event_f1", "event_recall",
    "n_true_events", "n_detected_events", "alarm_rate",
    "flag_everything_f1", "beats_flag_everything_event",
]].round(3)
event_view.to_csv(WORKING / "labelled_event_metrics.csv", index=False)
event_view

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

strategies = list(event_view.strategy)
colours = {"static": "#c1440e", "online-no-reset": "#8a8a8a",
           "periodic": "#1f6f8b", "drift-triggered": "#2a7f4f"}
baseline = float(event_view.flag_everything_f1.iloc[0])

x = np.arange(len(strategies))
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(x - 0.2, event_view.f1, width=0.38, color="#b0b0b0", label="point-wise F1")
ax.bar(x + 0.2, event_view.event_f1, width=0.38,
       color=[colours[s] for s in strategies], label="event-level F1")
ax.axhline(baseline, color="#d62728", ls="--", lw=1.1,
           label=f"flag-everything ({baseline:.3f})")
ax.set_xticks(x)
ax.set_xticklabels(strategies, fontsize=9)
ax.set_ylabel("F1")
ax.set_ylim(0, 0.75)
ax.set_title("Point-wise F1 buries the strategies at the baseline; "
             "event-level F1 separates them\n(skab-valve1)", fontsize=9, loc="left")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(WORKING / "labelled_event_f1.png", dpi=140)
plt.show()

### What event-level scoring shows

From a real run of exactly this code (matches `results/metrics.csv` in the repo):

| policy | point-wise F1 | event-level F1 | events caught |
| --- | --- | --- | --- |
| static | 0.513 | 0.513 | 15 / 15 |
| online-no-reset | 0.357 | 0.559 | 15 / 15 |
| periodic | 0.544 | **0.622** | 15 / 15 |
| drift-triggered | 0.502 | 0.593 | 15 / 15 |

1. **Point-wise F1 cannot separate the strategies.** All four sit near the 0.512
   flag-everything baseline, because point-wise recall is dominated by how many
   rows of each long fault block get flagged, not by whether the fault was caught.
2. **`online-no-reset` was never broken.** Its point-wise F1 of 0.357 reads as a
   failure, but it catches all 15 fault events; the low score was purely sparse
   flagging inside long blocks. At event level it clears the baseline.
3. **Every adaptive policy beats flag-everything at event level; `static` does
   not.** Flagging 99.8% of points, `static` earns nothing from counting blocks
   once -- correctly. `periodic` gains the most.
4. **Honest limit.** Every strategy catches all 15 events (event recall 1.0), so
   on valve1 event F1 reduces to a precision-driven ranking. That is the right
   operational picture -- they all find the faults, they differ only in
   false-alarm volume -- and it is exactly what point-wise F1 obscured. On NAB's
   short, sparse anomalies the recall axis does more work; see the transfer study
   (`scripts/run_transfer.py`) in the repo.